# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [3]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [5]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [6]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [7]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [8]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [9]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [11]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [12]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [13]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [14]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [15]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [16]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 15954.320, Val Loss: 12151.478


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 5423.495, Val Loss: 10948.160


In [17]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [18]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$128 $160 $23 $92 $21 $53 $73 $64 $18 $144 $165 $146 $43 $44 $48 $12 $10 $1 $22 $40 $23 $18 $24 $47 $175 $76 $230 $39 $85 $45 $134 $189 $80 $23 $83 $359 $36 $28 $96 $76 $139 $11 $20 $39 $28 $61 $36 $24 $49 $36 $27 $21 $87 $8 $57 $130 $31 $135 $26 $66 $105 $4 $41 $29 $395 $18 $41 $351 $5 $113 $13 $23 $183 $113 $13 $42 $98 $10 $12 $43 $79 $45 $41 $41 $5 $125 $14 $131 $35 $141 $14 $18 $8 $13 $39 $57 $13 $19 $208 $193 $12 $62 $17 $66 $13 $4 $46 $278 $22 $62 $32 $129 $87 $39 $65 $90 $36 $64 $32 $141 $29 $221 $40 $12 $89 $51 $12 $82 $31 $25 $51 $15 $33 $29 $71 $32 $66 $10 $49 $8 $1 $91 $38 $89 $89 $47 $24 $135 $17 $3 $11 $52 $3 $25 $41 $89 $98 $6 $26 $17 $65 $8 $2 $18 $296 $24 $4 $33 $17 $46 $28 $18 $298 $26 $28 $47 $40 $11 $20 $46 $15 $11 $87 $18 $22 $22 $58 $55 $2 $18 $43 $24 $1 $160 $2 $5 $81 $8 $6 $0 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [19]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [20]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [21]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [22]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [23]:
gpt_4__1_nano(test[0])

'$250'

In [24]:
test[0].price

219.0

In [25]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $34 $25 $10 $120 $90 $6 $65 $6 $870 $137 $71 $30 $9 $19 $8 $71 $4 $40 $31 $54 $26 $65 $25 $212 $303 $705 $5 $251 $64 $30 $90 $10 $50 $35 $119 $90 $31 $36 $18 $175 $45 $20 $105 $70 $0 $12 $3 $75 $52 $22 $105 $125 $0 $197 $16 $8 $59 $48 $3 $86 $18 $41 $40 $279 $9 $90 $295 $25 $74 $17 $8 $30 $6 $25 $11 $126 $0 $18 $3 $30 $3 $5 $64 $11 $0 $68 $56 $60 $1 $8 $20 $5 $20 $2 $78 $1 $93 $20 $425 $20 $3 $12 $41 $951 $18 $10 $380 $19 $99 $0 $36 $49 $53 $54 $230 $20 $5 $24 $47 $16 $161 $80 $16 $0 $10 $5 $51 $29 $59 $79 $13 $5 $0 $85 $0 $55 $10 $78 $62 $6 $100 $70 $9 $144 $118 $15 $690 $15 $13 $1 $44 $22 $20 $1 $129 $31 $41 $80 $5 $311 $17 $3 $2 $140 $2 $752 $25 $5 $5 $5 $3 $70 $8 $57 $201 $3 $7 $19 $57 $546 $15 $150 $99 $30 $8 $83 $3 $20 $2 $15 $69 $15 $11 $25 $70 $10 $120 $21 $1 

In [ ]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(claude_opus_4_5, test)

In [63]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3.1-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [64]:
evaluate(gemini_3_pro_preview, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$0 $24 $15 $20 $10 $40 $84 $65 $16 $80 $186 $20 $5 $6 $49 $3 $21 $20 $11 $29 $4 $4 $15 $25 $133 $204 $245 $3 $91 $60 $5 $30 $10 $58 $5 $169 $6 $37 $44 $13 $150 $30 $15 $105 $50 $5 $2 $2 $85 $2 $22 $108 $176 $0 $27 $19 $3 $10 $78 $5 $116 $48 $31 $75 $79 $10 $60 $305 $5 $16 $17 $2 $80 $3 $20 $17 $4 $1 $3 $6 $10 $3 $13 $69 $8 $25 $8 $116 $0 $11 $2 $40 $0 $5 $1 $133 $4 $22 $75 $245 $10 $27 $7 $69 $0 $72 $13 $340 $15 $100 $20 $35 $5 $73 $34 $0 $5 $5 $34 $17 $4 $161 $20 $76 $30 $25 $4 $21 $29 $89 $144 $17 $1 $5 $25 $3 $15 $35 $37 $2 $4 $50 $15 $5 $69 $7 $15 $85 $16 $13 $4 $44 $22 $40 $1 $21 $21 $41 $20 $0 $140 $17 $28 $2 $40 $7 $402 $20 $5 $0 $5 $3 $170 $13 $67 $1 $2 $27 $4 $4 $154 $10 $150 $89 $10 $3 $63 $7 $20 $8 $0 $6 $5 $61 $60 $0 $0 $80 $21 $11 

In [54]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [55]:
evaluate(gemini_2__5_flash_lite, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$1 $164 $5 $80 $20 $80 $79 $115 $4 $120 $363 $20 $15 $9 $19 $12 $101 $15 $240 $74 $16 $6 $20 $75 $142 $253 $145 $5 $151 $67 $15 $15 $10 $50 $0 $69 $60 $31 $34 $18 $135 $35 $5 $5 $100 $5 $7 $13 $70 $2 $28 $95 $25 $0 $67 $16 $8 $30 $72 $13 $86 $33 $46 $35 $229 $10 $59 $265 $25 $99 $14 $8 $30 $0 $25 $11 $51 $2 $8 $4 $20 $6 $5 $74 $18 $0 $168 $44 $20 $21 $3 $55 $5 $5 $4 $78 $11 $7 $120 $275 $15 $33 $2 $19 $24 $32 $13 $325 $14 $49 $5 $136 $39 $13 $4 $80 $0 $2 $64 $147 $9 $211 $50 $109 $30 $5 $5 $51 $29 $64 $179 $13 $10 $0 $95 $5 $25 $0 $43 $22 $21 $1 $20 $0 $14 $13 $20 $240 $85 $8 $4 $144 $12 $10 $6 $171 $26 $31 $20 $0 $90 $10 $63 $3 $90 $2 $752 $15 $15 $5 $0 $2 $220 $8 $37 $100 $6 $28 $86 $28 $54 $20 $250 $1 $25 $8 $58 $17 $10 $8 $30 $14 $15 $161 $35 $110 $59 $30 $6 $1 

In [30]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [40]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.5", messages=messages_for(item), reasoning_effort='none', seed=42)
    return response.choices[0].message.content


In [41]:
evaluate(gpt_5__1, test, size=50, workers=2)

  0%|          | 0/50 [00:00<?, ?it/s]

$10 $74 $5 $0 $10 $100 $84 $70 $6 $31 $14 $80 $0 $14 $39 $3 $11 $20 $10 $69 $26 $14 $25 $5 $87 $224 $166 $5 $91 $65 $5 $25 $170 $55 $74 $220 $30 $39 $44 $13 $160 $40 $20 $5 $40 $5 $7 $3 $65 $72 